# Your First LLM-Powered Tool

Starter notebook. Fill in the TODOs and **run every cell** before committing.
Do **not** commit your API key — load it from `.env`.


In [27]:
import os, json
import google.generativeai as genai


from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
assert GEMINI_API_KEY, "Set GEMINI_API_KEY"

genai.configure(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.5-flash-lite"

print(" Ready!")

 Ready!


## Task 1 — First calls and the sampling dial

Make a working call (print response **and token usage**), then send the same prompt 3× at temp ~0.1 and 3× at temp ~1.0.


In [7]:

model = genai.GenerativeModel(
    model_name=MODEL,
    system_instruction="You are a concise support assistant."
)

response = model.generate_content("My payment is not working, help!")

print(" answer:", response.text)
print(f" Input tokens:  {response.usage_metadata.prompt_token_count}")
print(f" Output tokens: {response.usage_metadata.candidates_token_count}")

prompt = "My payment is not working, help!"

for temp in [0.1, 1.0]:
    print(f"\n{'='*40}")
    print(f"  Temperature: {temp}")
    print('='*40)
    for i in range(3):
        r = model.generate_content(
            prompt,
            generation_config=genai.GenerationConfig(temperature=temp)
        )
        print(f"Run {i+1}: {r.text.strip()}")

 answer: I can help with that! To troubleshoot your payment issue, please tell me:

*   **What payment method are you trying to use?** (e.g., credit card, PayPal, etc.)
*   **What is the error message you are seeing?** (If any)
*   **What have you tried already?**
 Input tokens:  16
 Output tokens: 71

 Temperature: 0.1
Run 1: I'm sorry to hear you're having trouble with your payment. To help me assist you, please provide more details:

*   **What payment method are you trying to use?** (e.g., credit card, PayPal, bank transfer)
*   **What is the error message you are seeing, if any?**
*   **What were you trying to purchase or pay for?**
*   **Have you tried this payment method before successfully?**
Run 2: I'm sorry to hear you're having trouble with your payment. To help me assist you, please provide more details:

*   **What payment method are you trying to use?** (e.g., credit card, PayPal, bank transfer)
*   **What is the specific error message you are seeing?**
*   **What were yo

**What changed, and why?**

## What changed, and why?

**Temperature 0.1 (low):**
- All 3 runs are nearly identical
- Same structure, same questions, same order
- The model almost always picks the most likely token → repeatable output

**Temperature 1.0 (high):**
- Each run is noticeably different
- Sometimes bullet list, sometimes numbered list
- Different questions, different order
- The model takes more chances on less likely tokens → varied output

**Conclusion:**
Low temperature (0.1) is ideal for classification and extraction tasks
where consistency matters. High temperature (1.0) suits creative tasks
where variety is desirable. This matches the sampling theory from the
lesson: low temp sharpens the distribution, high temp flattens it.


## Note: Switched to Local Ollama Fallback

Gemini free tier quota was exhausted during Task 2.
Switching to local Ollama (llama3.2:3b) to continue without interruption.

This is exactly the hosted-vs-local trade-off covered in the lesson:
Gemini is more capable but rate-limited; Ollama is free, private, and always available.


## Task 1 — Observations: Temperature and Sampling

### What changed between temperature 0.1 and 1.0?

At **temperature 0.1**, all three runs produced nearly identical responses.
The structure, wording, and list of questions were almost the same every time.
This happens because low temperature sharpens the probability distribution —
the model almost always picks the highest-probability token, making output
deterministic and repetitive.

At **temperature 1.0**, each run was noticeably different.
The opening phrase changed, the number of follow-up questions varied,
and once the model used a numbered list instead of bullet points.
High temperature flattens the distribution — the model samples from a
wider range of tokens, introducing variety and unpredictability.

### Connection to sampling theory

In the lesson, temperature controls how "peaked" or "flat" the token
probability distribution is. Low temperature → peaked → model picks
the same top token every time → consistent output. High temperature →
flat → model picks less likely tokens too → varied output.

For a support classifier, **low temperature is correct** — you want
the same ticket to always get the same label. High temperature would
be useful for creative tasks like generating response suggestions.

## Task 2 — Prompt-pattern bake-off

Classify the tickets three ways (zero-shot, few-shot, chain-of-thought) into `billing / bug / feature_request / other`. Build a comparison table. Record prompts + verdict in `prompts.md`.


In [21]:
import json
import pandas as pd
import ollama

with open("tickets.json") as f:
    tickets = json.load(f)

LABELS = ["billing", "bug", "feature_request", "other"]

def classify(text, style):
    
    if style == "zero_shot":
        prompt = f"""Classify this support ticket into one of: billing, bug, feature_request, other.
Reply with only the label.

Ticket: {text}"""

    elif style == "few_shot":
        prompt = f"""Classify support tickets into: billing, bug, feature_request, other.

Ticket: "I was charged twice this month." → billing
Ticket: "The app crashes when I upload a file." → bug
Ticket: "Can you add dark mode?" → feature_request
Ticket: "Where is my order confirmation?" → other

Ticket: "{text}" →"""

    elif style == "cot":
        prompt = f"""Classify this support ticket into: billing, bug, feature_request, other.
Think step by step, then give the final label.

Ticket: {text}
Answer:"""

    r = ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    return r["message"]["content"].strip().lower()


# Bütün ticketləri 3 üsulla classify et
results = []
for ticket in tickets:
    text = ticket if isinstance(ticket, str) else ticket.get("text", ticket.get("message", str(ticket)))
    row = {
        "ticket": text[:60],
        "zero_shot": classify(text, "zero_shot"),
        "few_shot":  classify(text, "few_shot"),
        "cot":       classify(text, "cot"),
    }
    results.append(row)
    print(f"✅ {row['ticket'][:40]}...")

df = pd.DataFrame(results)
print(df.to_markdown(index=False))

 I was charged twice for my subscription ...
 The export button throws a 500 error eve...
 It would be great if you could add a dar...
 How do I reset my password? I can't find...
 The app crashes on startup after the lat...
 Can you send me a copy of my last invoic...
 Please add PDF export â€” CSV alone isn'...
 Just wanted to say the new UI looks real...
| ticket                                                       | zero_shot       | few_shot                                                                                                                                                                                                                                                 | cot                                                                                                                                                                                                                                                                                                               

## Bake-off Verdict

**Zero-shot:** Clean, one-word answers. Fast and mostly correct.
Failed on "password reset" → labeled as feature_request instead of other.

**Few-shot:** Correct labels but too verbose — returned explanations
instead of just the label. The small local model (llama3.2:3b) did not
follow the output format strictly.

**Chain-of-thought:** Reasoning was logical but answers were even more
verbose. Correct on most tickets but slow and hard to parse.

**Winner: Zero-shot** for this task — simple classification needs
a clean label, not an essay. Few-shot would win with a larger model
that better follows format instructions.

**Where each failed:**
- Zero-shot: ambiguous tickets (password reset → wrong label)
- Few-shot: format not followed, too much explanation
- CoT: correct reasoning but unparseable output for a pipeline

## Task 3 — Structured output as a function

Return JSON `{label, confidence, reason}` with low temperature, then **parse and validate** it. Handle one bad-output case. Re-run against local Ollama and compare.


In [24]:
import json
import ollama

ALLOWED_LABELS = ["billing", "bug", "feature_request", "other"]

def classify_structured(text):
    prompt = f"""Return ONLY a JSON object. No explanation. No markdown.

Example output:
{{"label": "billing", "confidence": 0.85, "reason": "user was charged incorrectly"}}

Rules:
- label must be exactly one of: billing, bug, feature_request, other
- confidence must be a number between 0.0 and 1.0
- reason must be a short string

Ticket: {text}"""

    r = ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw = r["message"]["content"].strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    
    try:
        data = json.loads(raw)
    except json.JSONDecodeError:
        print(f"⚠️  Invalid JSON: {raw[:80]}")
        return None
    
    if data.get("label") not in ALLOWED_LABELS:
        print(f"⚠️  Invalid label: {data.get('label')}")
        return None
    
    if not 0.0 <= float(data.get("confidence", -1)) <= 1.0:
        print(f"⚠️  Invalid confidence: {data.get('confidence')}")
        return None
    
    return data


# Test et
test_tickets = [
    "I was charged twice this month.",
    "App crashes on startup.",
    "Please add dark mode.",
    "Just wanted to say great work!",
]

for ticket in test_tickets:
    result = classify_structured(ticket)
    if result:
        print(f"✅ {ticket[:40]}")
        print(f"   label: {result['label']} | confidence: {result['confidence']} | reason: {result['reason']}")
    else:
        print(f"❌ Failed: {ticket[:40]}")

✅ I was charged twice this month.
   label: billing | confidence: 0.9 | reason: user was charged incorrectly
✅ App crashes on startup.
   label: bug | confidence: 0.97 | reason: app crashes on startup
✅ Please add dark mode.
   label: feature_request | confidence: 0.99 | reason: Please add dark mode
✅ Just wanted to say great work!
   label: other | confidence: 0.0 | reason: user was highly satisfied


**Local vs hosted: did the small local model produce valid JSON as reliably?**

The local Ollama model (llama3.2:3b) initially failed to produce valid JSON —
it returned the schema description literally instead of filling in real values.
After making the prompt more explicit with a concrete example output, the model
returned valid, parseable JSON reliably. Compared to Gemini, the local model
required more prompt engineering to follow format instructions, but once guided
correctly it performed well on this simple classification task.